# BrainTumNet Phase 2 – Full Pipeline Reference

Notebook này gom toàn bộ code triển khai phương pháp BrainTumNet Phase 2 từ xử lý dữ liệu đến huấn luyện, dùng làm mã tham chiếu cho paper.

## Lộ trình trong notebook

1. Thiết lập chung & cấu hình chuẩn
2. Pipeline xử lý dữ liệu BraTS (NIfTI → PNG đa lớp)
3. Chuyển đổi dataset sang LMDB tốc độ cao
4. Dataset + augmentation + DataLoader
5. Kiến trúc BrainTumNet V2 (SegUNetV2 + ROI-guided classifier)
6. Losses, metrics và logger
7. Huấn luyện end-to-end (5-fold, AMP, scheduler, checkpoint)
8. Ví dụ sử dụng


In [ ]:
from scipy.ndimage import distance_transform_edt

In [ ]:
import os
import math
import json
import time
import random
import shutil
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image
from sklearn.model_selection import KFold
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False

try:
    import lmdb
except ImportError:
    lmdb = None

from scipy.ndimage import gaussian_filter, map_coordinates, zoom

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "braintumnet" / "data"
PROCESSED_DIR = DATA_ROOT / "processed_multiclass_full"
LMDB_DIR = DATA_ROOT / "lmdb_processed_multiclass_full"
LOG_ROOT = PROJECT_ROOT / "logs"
RUN_ROOT = PROJECT_ROOT / "runs"
CKPT_ROOT = PROJECT_ROOT / "checkpoints"

print(f"Project root: {PROJECT_ROOT}")

## 1. Cấu hình chung

In [ ]:
@dataclass
class PreprocessConfig:
    nifti_dir: Path
    out_dir: Path = PROCESSED_DIR
    img_size: int = 256
    slices_per_case: Optional[int] = None
    tumor_ratio: float = 0.7
    num_folds: int = 5
    seed: int = 42
    max_cases: Optional[int] = None

@dataclass
class LmdbConversionConfig:
    input_dir: Path
    output_dir: Path = LMDB_DIR
    map_size_gb: int = 50
    verify: bool = True


def build_phase2_config(
    processed_root: Path = PROCESSED_DIR,
    lmdb_root: Path = LMDB_DIR,
    backend: str = "lmdb",
    model_size: str = "large",
) -> Dict[str, Any]:
    base_channels = 64 if model_size == "large" else 48
    dim = 512 if model_size == "large" else 384

    cfg = {
        "data": {
            "backend": backend,
            "raw_root": str(DATA_ROOT / "raw"),
            "proc_root": str(processed_root),
            "lmdb_root": str(lmdb_root),
            "modality": "multi",
            "img_size": 256,
            "slices_per_case": 155,
            "tumor_slice_ratio": 0.7,
            "num_folds": 5,
            "fold": 0,
            "cache_size": 0,
        },
        "train": {
            "epochs": 400,
            "batch_size": 12,
            "val_batch_size": 24,
            "lr": 5e-5,
            "weight_decay": 1.5e-4,
            "workers": 12,
            "loss_type": "dice_focal",
            "seg_loss_weight": 1.0,
            "cls_loss_weight": 0.5,
            "dice_weight": 1.0,
            "focal_weight": 1.0,
            "iou_weight": 2.5,
            "boundary_weight": 0.0,
            "focal_alpha": [0.0, 0.4, 0.3],
            "focal_gamma": 3.0,
            "class_weights": [1.0, 3.0, 4.0],
            "ignore_background": True,
            "aux_weight": 0.3,
            "optimizer": "adamw",
            "optimizer_fused": False,
            "grad_clip_norm": 1.0,
            "scheduler": "cosine",
            "warmup_steps": 2000,
            "min_lr": 5e-7,
            "amp": True,
            "amp_dtype": "float16",
            "grad_accum_steps": 1,
            "channels_last": False,
            "cudnn_benchmark": True,
            "pin_memory": True,
            "prefetch_factor": 4,
            "persistent_workers": True,
            "early_stop_patience": 30,
            "val_interval": 1,
            "log_interval": 10,
            "save_interval": 10,
            "aux_weight_initial": 0.5,
            "aux_weight_final": 0.1,
        },
        "model": {
            "model_type": "segunetv2",
            "in_channels": 4,
            "num_classes_seg": 3,
            "num_classes_cls": 2,
            "base": base_channels,
            "dim": dim,
            "patch_size": 8,
            "depth": 4,
            "n_heads": 8,
            "dropout": 0.2,
            "roi_stop_grad": True,
            "deep_supervision": True,
            "multi_scale_fusion": True,
            "boundary_refinement": False,
            "use_multiscale_transformer": True,
            "use_attention_gates": True,
        },
        "logging": {
            "log_dir": str(LOG_ROOT),
            "out_dir": str(RUN_ROOT),
            "save_dir": str(CKPT_ROOT),
            "exp_name": "braintumnet_v2",
            "use_tensorboard": True,
            "log_every_n_steps": 50,
            "save_top_k": 3,
        },
        "augment": {
            "rotate_deg": 45,
            "hflip_p": 0.5,
            "vflip_p": 0.5,
            "brightness_range": [0.75, 1.25],
            "contrast_range": [0.75, 1.25],
            "gamma_range": [0.85, 1.15],
            "gaussian_noise_p": 0.2,
            "gaussian_noise_std": 0.01,
            "elastic_deform_p": 0.3,
            "elastic_alpha": 30,
            "elastic_sigma": 4,
            "bias_field_p": 0.5,
            "bias_field_scale": 0.3,
            "gaussian_blur_p": 0.2,
            "gaussian_blur_sigma": [0.5, 1.5],
            "gamma_p": 0.5,
            "cutout_p": 0.2,
            "cutout_n_holes": 3,
            "cutout_size": 20,
            "local_shuffle_p": 0.15,
            "local_shuffle_size": 3,
        },
    }
    return cfg

## 2. Pipeline xử lý dữ liệu BraTS (NIfTI → PNG đa lớp)

In [ ]:
def set_seed(seed: int, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

def load_nifti_volume(case_dir: Path, modality: str) -> np.ndarray:
    nii_file = case_dir / f"{case_dir.name}_{modality}.nii"
    if not nii_file.exists():
        raise FileNotFoundError(f"Missing {modality}: {nii_file}")
    return nib.load(str(nii_file)).get_fdata()

def convert_brats_seg_to_3class(seg_slice: np.ndarray) -> np.ndarray:
    mask = np.zeros_like(seg_slice, dtype=np.uint8)
    mask[seg_slice == 2] = 2
    mask[(seg_slice == 1) | (seg_slice == 4)] = 1
    return mask

def normalize_slice(img_slice: np.ndarray) -> np.ndarray:
    brain_mask = img_slice > 0
    if brain_mask.sum() == 0:
        return np.zeros_like(img_slice, dtype=np.uint8)
    p1 = np.percentile(img_slice[brain_mask], 1)
    p99 = np.percentile(img_slice[brain_mask], 99)
    img_clipped = np.clip(img_slice, p1, p99)
    img_norm = (img_clipped - p1) / (p99 - p1 + 1e-8)
    return (img_norm * 255).astype(np.uint8)

def resize_array(arr: np.ndarray, target_size: int, is_mask: bool) -> np.ndarray:
    img = Image.fromarray(arr)
    resample = Image.NEAREST if is_mask else Image.BILINEAR
    return np.array(img.resize((target_size, target_size), resample))

def select_slices_with_tumor(seg_volume: np.ndarray, slices_per_case: Optional[int], tumor_ratio: float) -> List[int]:
    total = seg_volume.shape[2]
    if slices_per_case is None or slices_per_case >= total:
        return list(range(total))
    tumor, non_tumor = [], []
    for z in range(total):
        (tumor if seg_volume[:, :, z].sum() > 0 else non_tumor).append(z)
    n_tumor = min(len(tumor), int(slices_per_case * tumor_ratio))
    n_non = slices_per_case - n_tumor
    if n_tumor > 0 and tumor:
        tumor_idx = np.linspace(tumor[0], tumor[-1], n_tumor).astype(int).tolist()
    else:
        tumor_idx = []
    if n_non > 0 and non_tumor:
        step = max(1, len(non_tumor) // max(1, n_non))
        non_idx = non_tumor[::step][:n_non]
    else:
        non_idx = []
    return sorted(set(tumor_idx + non_idx))

def process_case(case_dir: Path, cfg: PreprocessConfig) -> List[Dict[str, Any]]:
    try:
        flair = load_nifti_volume(case_dir, 'flair')
        t1 = load_nifti_volume(case_dir, 't1')
        t1ce = load_nifti_volume(case_dir, 't1ce')
        t2 = load_nifti_volume(case_dir, 't2')
        seg = load_nifti_volume(case_dir, 'seg')
    except FileNotFoundError as exc:
        print(f"Skip {case_dir.name}: {exc}")
        return []
    selected = select_slices_with_tumor(seg, cfg.slices_per_case, cfg.tumor_ratio)
    records = []
    for z in selected:
        slice_id = f"{case_dir.name}_slice{z:03d}"
        modalities = {'flair': flair[:, :, z], 't1': t1[:, :, z], 't1ce': t1ce[:, :, z], 't2': t2[:, :, z]}
        seg_slice = seg[:, :, z]
        seg_3c = convert_brats_seg_to_3class(seg_slice)
        for name, vol in modalities.items():
            norm = normalize_slice(vol)
            resized = resize_array(norm, cfg.img_size, is_mask=False)
            out_dir = cfg.out_dir / name
            out_dir.mkdir(parents=True, exist_ok=True)
            Image.fromarray(resized).save(out_dir / f"{slice_id}.png")
        seg_resized = resize_array(seg_3c, cfg.img_size, is_mask=True)
        seg_dir = cfg.out_dir / "seg"
        seg_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(seg_resized, mode='L').save(seg_dir / f"{slice_id}.png")
        has_tc = int((seg_resized == 1).any())
        has_ed = int((seg_resized == 2).any())
        records.append({
            'slice_id': slice_id,
            'case_id': case_dir.name,
            'slice_idx': z,
            'has_wt': int(has_tc or has_ed),
            'has_tc': has_tc,
            'has_ed': has_ed,
            'label': 'WT' if (has_tc or has_ed) else 'Normal'
        })
    return records

def create_kfold_splits(df: pd.DataFrame, num_folds: int, seed: int) -> List[tuple]:
    cases = df['case_id'].unique()
    splitter = KFold(n_splits=num_folds, shuffle=True, random_state=seed)
    splits = []
    for train_idx, val_idx in splitter.split(cases):
        train_cases = cases[train_idx]
        val_cases = cases[val_idx]
        train_rows = df[df['case_id'].isin(train_cases)].index.tolist()
        val_rows = df[df['case_id'].isin(val_cases)].index.tolist()
        splits.append((train_rows, val_rows))
    return splits

def preprocess_brats_to_png(cfg: PreprocessConfig) -> pd.DataFrame:
    set_seed(cfg.seed)
    cfg.out_dir.mkdir(parents=True, exist_ok=True)
    case_dirs = sorted([d for d in cfg.nifti_dir.iterdir() if d.is_dir()])
    if cfg.max_cases:
        case_dirs = case_dirs[:cfg.max_cases]
    print(f"Processing {len(case_dirs)} cases → {cfg.out_dir}")
    records = []
    for case_dir in tqdm(case_dirs, desc='Cases'):
        records.extend(process_case(case_dir, cfg))
    df = pd.DataFrame(records)
    if df.empty:
        raise RuntimeError("Không tạo được lát cắt nào")
    df.to_csv(cfg.out_dir / "all_slices.csv", index=False)
    print(f"Saved {len(df)} slices from {df['case_id'].nunique()} cases")
    if cfg.num_folds > 1 and df['case_id'].nunique() >= cfg.num_folds:
        splits = create_kfold_splits(df, cfg.num_folds, cfg.seed)
        for fold, (train_idx, val_idx) in enumerate(splits):
            df.iloc[train_idx].to_csv(cfg.out_dir / f"train_fold{fold}.csv", index=False)
            df.iloc[val_idx].to_csv(cfg.out_dir / f"val_fold{fold}.csv", index=False)
            print(f"Fold {fold}: train {len(train_idx)}, val {len(val_idx)}")
    mapping_df = df[['slice_id', 'case_id']].sort_values('slice_id')
    mapping_df.to_csv(cfg.out_dir / "mapping.csv", index=False)
    name_mapping = cfg.nifti_dir / "name_mapping.csv"
    case_labels = []
    if name_mapping.exists():
        grades = pd.read_csv(name_mapping)
        grade_map = {row['BraTS_2020_subject_ID']: 0 if row['Grade'] == 'HGG' else 1 for _, row in grades.iterrows()}
        for case_id in sorted(df['case_id'].unique()):
            case_labels.append({'case_id': case_id, 'label': grade_map.get(case_id, 0)})
    else:
        for case_id in sorted(df['case_id'].unique()):
            case_labels.append({'case_id': case_id, 'label': 0})
    pd.DataFrame(case_labels).to_csv(cfg.out_dir / "labels.csv", index=False)
    class_mapping = {
        "num_classes": 3,
        "class_names": ["Background", "TumorCore", "Edema"],
        "regions": {"WT": "Whole Tumor = TC + ED", "TC": "Tumor Core", "ED": "Edema"},
        "brats_label_mapping": {"0": "Background", "1": "NCR/NET", "2": "Edema", "4": "Enhancing"}
    }
    with open(cfg.out_dir / "class_mapping.json", 'w') as f:
        json.dump(class_mapping, f, indent=2)
    print("Preprocessing complete")
    return df

In [ ]:
def load_multimodal_sample(input_dir: Path, slice_id: str) -> Dict[str, Any]:
    def read_png(modality: str):
        return np.array(Image.open(input_dir / modality / f"{slice_id}.png"))
    flair = read_png('flair')
    t1 = read_png('t1')
    t1ce = read_png('t1ce')
    t2 = read_png('t2')
    seg = np.array(Image.open(input_dir / 'seg' / f"{slice_id}.png"))
    image = np.stack([flair, t1, t1ce, t2], axis=0).astype(np.uint8)
    return {'image': image, 'mask': seg.astype(np.uint8), 'slice_id': slice_id}

def iter_slice_ids(input_dir: Path) -> List[str]:
    flair_dir = input_dir / 'flair'
    if not flair_dir.exists():
        raise FileNotFoundError(f"Missing flair folder: {flair_dir}")
    return sorted(p.stem for p in flair_dir.glob('*.png'))

def convert_png_to_lmdb(cfg: LmdbConversionConfig):
    if lmdb is None:
        raise ImportError("lmdb chưa được cài (pip install lmdb)")
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    slice_ids = iter_slice_ids(cfg.input_dir)
    print(f"Converting {len(slice_ids)} slices → {cfg.output_dir}")
    env = lmdb.open(str(cfg.output_dir), map_size=cfg.map_size_gb * (1024 ** 3), readonly=False, meminit=False, map_async=True)
    metadata = {
        'num_samples': len(slice_ids),
        'modalities': ['flair', 't1', 't1ce', 't2'],
        'num_channels': 4,
        'num_classes': 3,
        'slice_ids': slice_ids,
    }
    with env.begin(write=True) as txn:
        for idx, slice_id in enumerate(tqdm(slice_ids, desc='LMDB')):
            sample = load_multimodal_sample(cfg.input_dir, slice_id)
            txn.put(f"{idx:08d}".encode('ascii'), pickle.dumps(sample, protocol=pickle.HIGHEST_PROTOCOL))
        txn.put(b'__metadata__', pickle.dumps(metadata))
    env.sync(); env.close()
    for csv in cfg.input_dir.glob('*.csv'):
        shutil.copy(csv, cfg.output_dir / csv.name)
    meta_out = metadata.copy(); meta_out['slice_ids'] = f"<{len(slice_ids)} items>"
    with open(cfg.output_dir / 'meta.json', 'w') as f:
        json.dump(meta_out, f, indent=2)
    print("LMDB conversion done")
    if cfg.verify:
        verify_lmdb_database(cfg.output_dir)

def verify_lmdb_database(output_dir: Path):
    if lmdb is None:
        raise ImportError("lmdb chưa được cài")
    env = lmdb.open(str(output_dir), readonly=True, lock=False)
    with env.begin() as txn:
        metadata = pickle.loads(txn.get(b'__metadata__'))
        summary = {k: (f"<{len(v)} items>" if k == 'slice_ids' else v) for k, v in metadata.items()}
        print("Metadata:", json.dumps(summary, indent=2))
        sample = pickle.loads(txn.get(b'00000000'))
        print(f"Sample {sample['slice_id']} image {sample['image'].shape} mask {sample['mask'].shape}")
    env.close()

In [ ]:
class SliceDataset(Dataset):
    def __init__(self, proc_root: Path, split_file: Path, cfg: Dict[str, Any], train: bool):
        self.proc_root = proc_root
        self.train = train
        self.cfg = cfg
        if split_file.suffix == '.csv':
            df = pd.read_csv(split_file)
            self.slice_ids = df['slice_id'].tolist()
        else:
            with open(split_file) as f:
                self.slice_ids = [line.strip() for line in f if line.strip()]
        labels_csv = proc_root / 'labels.csv'
        self.case_label = {}
        if labels_csv.exists():
            df = pd.read_csv(labels_csv)
            self.case_label = dict(zip(df['case_id'], df['label']))
        mapping_csv = proc_root / 'mapping.csv'
        self.slice_case = {}
        if mapping_csv.exists():
            df = pd.read_csv(mapping_csv)
            self.slice_case = dict(zip(df['slice_id'], df['case_id']))

    def __len__(self):
        return len(self.slice_ids)

    def _load_modality(self, modality: str, slice_id: str) -> torch.Tensor:
        path = self.proc_root / modality / f"{slice_id}.png"
        img = Image.open(path).convert('L')
        tensor = to_tensor01(img)
        return tensor

    def __getitem__(self, idx: int):
        sid = self.slice_ids[idx]
        img = torch.stack([
            self._load_modality('flair', sid),
            self._load_modality('t1', sid),
            self._load_modality('t1ce', sid),
            self._load_modality('t2', sid)
        ], dim=0)
        mask = torch.from_numpy(np.array(Image.open(self.proc_root / 'seg' / f"{sid}.png"))).long().unsqueeze(0)
        cid = self.slice_case.get(sid, sid.split('_')[0])
        label = self.case_label.get(cid, 0)
        return {
            'image': img.float(),
            'mask': mask,
            'label': torch.tensor(label, dtype=torch.long),
            'slice_id': sid,
            'case_id': cid
        }

class LMDBDataset(Dataset):
    def __init__(self, lmdb_root: Path, split_file: Path, cfg: Dict[str, Any], train: bool):
        if lmdb is None:
            raise ImportError("lmdb chưa được cài")
        self.train = train
        self.cfg = cfg
        self.lmdb_root = lmdb_root
        env = lmdb.open(str(lmdb_root), readonly=True, lock=False, readahead=False, meminit=False)
        with env.begin() as txn:
            metadata = pickle.loads(txn.get(b'__metadata__'))
            self.slice_ids_all = metadata['slice_ids']
        env.close()
        if split_file.suffix == '.csv':
            df = pd.read_csv(split_file)
            target_ids = df['slice_id'].tolist()
        else:
            with open(split_file) as f:
                target_ids = [line.strip() for line in f if line.strip()]
        self.slice_to_idx = {sid: i for i, sid in enumerate(self.slice_ids_all)}
        self.indices = [self.slice_to_idx[sid] for sid in target_ids if sid in self.slice_to_idx]
        labels_csv = lmdb_root / 'labels.csv'
        self.case_label = {}
        if labels_csv.exists():
            df = pd.read_csv(labels_csv)
            self.case_label = dict(zip(df['case_id'], df['label']))
        mapping_csv = lmdb_root / 'mapping.csv'
        self.slice_case = {}
        if mapping_csv.exists():
            df = pd.read_csv(mapping_csv)
            self.slice_case = dict(zip(df['slice_id'], df['case_id']))
        self.env = None
        if train:
            keys = [
                'elastic_deform_p','elastic_alpha','elastic_sigma','bias_field_p','bias_field_scale',
                'gaussian_blur_p','gaussian_blur_sigma','gamma_p','gamma_range',
                'cutout_p','cutout_n_holes','cutout_size','local_shuffle_p','local_shuffle_size'
            ]
            aug_cfg = {k: cfg['augment'][k] for k in keys if k in cfg['augment']}
            self.medical_aug = MedicalAugmentation(**aug_cfg) if aug_cfg else None
        else:
            self.medical_aug = None

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx: int):
        if self.env is None:
            self.env = lmdb.open(str(self.lmdb_root), readonly=True, lock=False, readahead=True, meminit=False)
        lmdb_idx = self.indices[idx]
        with self.env.begin() as txn:
            sample = pickle.loads(txn.get(f"{lmdb_idx:08d}".encode('ascii')))
        img = torch.from_numpy(sample['image']).float()
        mask = torch.from_numpy(sample['mask']).long().unsqueeze(0)
        if self.train and self.medical_aug is not None:
            img, mask = self.medical_aug(img, mask)
        cid = self.slice_case.get(sample['slice_id'], sample['slice_id'].split('_')[0])
        label = self.case_label.get(cid, 0)
        return {
            'image': img,
            'mask': mask,
            'label': torch.tensor(label, dtype=torch.long),
            'slice_id': sample['slice_id'],
            'case_id': cid
        }

def create_dataset(cfg: Dict[str, Any], fold: int, train: bool):
    backend = cfg['data'].get('backend', 'lmdb')
    root = Path(cfg['data']['proc_root'] if backend == 'png' else cfg['data']['lmdb_root'])
    split = root / ('train_fold{}.csv'.format(fold) if train else 'val_fold{}.csv'.format(fold))
    if backend == 'png':
        return SliceDataset(root, split, cfg, train)
    elif backend == 'lmdb':
        return LMDBDataset(root, split, cfg, train)
    raise ValueError(f"Unknown backend {backend}")

def build_dataloaders(cfg: Dict[str, Any], fold: int):
    train_ds = create_dataset(cfg, fold, True)
    val_ds = create_dataset(cfg, fold, False)
    loader_kwargs = dict(
        num_workers=cfg['train']['workers'],
        pin_memory=cfg['train'].get('pin_memory', True),
        persistent_workers=cfg['train'].get('persistent_workers', True)
    )
    train_loader = DataLoader(train_ds, batch_size=cfg['train']['batch_size'], shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, batch_size=cfg['train'].get('val_batch_size', cfg['train']['batch_size']), shuffle=False, **loader_kwargs)
    return train_loader, val_loader

## 5. Kiến trúc BrainTumNet V2

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
        )

    def forward(self, x):
        attn = torch.sigmoid(self.mlp(self.avg(x)) + self.mlp(self.max(x)))
        return x * attn

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        attn = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], dim=1)
        attn = torch.sigmoid(self.conv(attn))
        return x * attn

class CBAM(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, kernel_size: int = 7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.sa(self.ca(x))

class PatchEmbed(nn.Module):
    def __init__(self, in_ch: int, embed_dim: int, patch: int):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch, stride=patch)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        tokens = self.norm(tokens)
        return tokens, (H, W)

class SoftMaskGenerator(nn.Module):
    def __init__(self, dim: int, hidden: int = 128, n_heads: int = 4):
        super().__init__()
        self.n_heads = n_heads
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(),
            nn.Linear(hidden, n_heads), nn.Sigmoid()
        )

    def forward(self, tokens):
        mask = self.mlp(tokens)
        return mask.permute(0, 2, 1).contiguous()

class MaskedSelfAttention(nn.Module):
    def __init__(self, dim: int, n_heads: int = 4):
        super().__init__()
        self.n_heads = n_heads
        self.dim = dim
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.use_sdpa = hasattr(F, 'scaled_dot_product_attention')

    def forward(self, x, softmask):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if self.use_sdpa and torch.allclose(softmask, torch.ones_like(softmask)):
            out = F.scaled_dot_product_attention(q, k, v)
            out = out.transpose(1, 2).reshape(B, N, C)
        else:
            attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            bias = torch.log(softmask.unsqueeze(-2) + 1e-6)
            attn = F.softmax(attn + bias, dim=-1)
            out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(out)

class MaskedTransformerBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int = 4, mlp_ratio: float = 4.0, drop: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MaskedSelfAttention(dim, n_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, dim), nn.Dropout(drop)
        )

    def forward(self, x, softmask):
        x = x + self.attn(self.norm1(x), softmask)
        x = x + self.mlp(self.norm2(x))
        return x

class AdaptiveMaskedTransformer(nn.Module):
    def __init__(self, in_ch: int, dim: int, patch_size: int = 8, depth: int = 2, n_heads: int = 4):
        super().__init__()
        self.pe = PatchEmbed(in_ch, dim, patch_size)
        self.mask_gen = SoftMaskGenerator(dim, hidden=dim // 2, n_heads=n_heads)
        self.blocks = nn.ModuleList([MaskedTransformerBlock(dim, n_heads) for _ in range(depth)])

    def forward(self, x):
        tokens, (H, W) = self.pe(x)
        softmask = self.mask_gen(tokens)
        for blk in self.blocks:
            tokens = blk(tokens, softmask)
        feat = tokens.transpose(1, 2).reshape(x.size(0), tokens.size(-1), H, W)
        return feat

class TransformerBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int = 8, mlp_ratio: float = 4.0, drop: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, dim), nn.Dropout(drop)
        )

    def forward(self, x):
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class MultiScalePatchEmbed(nn.Module):
    def __init__(self, in_ch: int, embed_dim: int, patch_sizes=(4, 8, 16)):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Conv2d(in_ch, embed_dim, kernel_size=p, stride=p) for p in patch_sizes
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in patch_sizes])
        self.patch_sizes = patch_sizes

    def forward(self, x):
        tokens = []
        shapes = []
        for proj, norm in zip(self.proj, self.norms):
            patch = proj(x)
            B, C, H, W = patch.shape
            tok = patch.flatten(2).transpose(1, 2)
            tok = norm(tok)
            tokens.append(tok)
            shapes.append((H, W))
        return tokens, shapes

class MultiScaleTransformerBottleneck(nn.Module):
    def __init__(self, in_ch: int, dim: int, patch_sizes=(4, 8, 16), depth: int = 4, n_heads: int = 8):
        super().__init__()
        self.patch_embed = MultiScalePatchEmbed(in_ch, dim, patch_sizes)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads) for _ in range(depth)])
        self.fusion = nn.Sequential(
            nn.Linear(dim * len(patch_sizes), dim),
            nn.LayerNorm(dim),
            nn.GELU()
        )

    def forward(self, x):
        tokens_per_scale, shapes = self.patch_embed(x)
        processed = []
        for tokens in tokens_per_scale:
            for blk in self.blocks:
                tokens = blk(tokens)
            processed.append(tokens)
        target_shape = shapes[0]
        upsampled = []
        for tokens, (h, w) in zip(processed, shapes):
            if (h, w) != target_shape:
                spatial = tokens.transpose(1, 2).reshape(x.size(0), -1, h, w)
                spatial = F.interpolate(spatial, size=target_shape, mode='bilinear', align_corners=False)
                tokens = spatial.flatten(2).transpose(1, 2)
            upsampled.append(tokens)
        fused = torch.cat(upsampled, dim=-1)
        fused = self.fusion(fused)
        spatial = fused.transpose(1, 2).reshape(x.size(0), -1, target_shape[0], target_shape[1])
        return F.interpolate(spatial, size=x.shape[2:], mode='bilinear', align_corners=False)

In [ ]:
def conv_norm_act(in_ch, out_ch, k=3, s=1, p=1, norm='instance', dropout=0.0):
    layers = [nn.Conv2d(in_ch, out_ch, k, s, p, bias=False)]
    if norm == 'instance':
        layers.append(nn.InstanceNorm2d(out_ch, affine=True))
    elif norm == 'batch':
        layers.append(nn.BatchNorm2d(out_ch))
    else:
        layers.append(nn.GroupNorm(min(32, out_ch // 4), out_ch))
    layers.append(nn.LeakyReLU(0.01, inplace=True))
    if dropout > 0:
        layers.append(nn.Dropout2d(dropout))
    return nn.Sequential(*layers)

class ResidualConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm='instance', dropout=0.0):
        super().__init__()
        self.conv1 = conv_norm_act(in_ch, out_ch, norm=norm, dropout=dropout)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True) if norm == 'instance' else nn.BatchNorm2d(out_ch)
        )
        self.residual = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, x):
        identity = self.residual(x)
        out = self.conv1(x)
        out = self.conv2(out)
        return self.act(out + identity)

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm='instance', dropout=0.0):
        super().__init__()
        self.block = ResidualConvBlock(in_ch, out_ch, norm=norm, dropout=dropout)
        self.downsample = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1, bias=False)

    def forward(self, x):
        feat = self.block(x)
        down = self.downsample(feat)
        return feat, down

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l):
        super().__init__()
        F_int = F_l // 2
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=False), nn.InstanceNorm2d(F_int, affine=True))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=False), nn.InstanceNorm2d(F_int, affine=True))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=True), nn.Sigmoid())
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, g, x):
        psi = self.act(self.W_g(g) + self.W_x(x))
        psi = self.psi(psi)
        return x * psi

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm='instance', dropout=0.0, use_attention_gate=False):
        super().__init__()
        self.use_attention_gate = use_attention_gate
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2, bias=False)
        if use_attention_gate:
            self.attn = AttentionGate(out_ch, out_ch)
        self.cbam = CBAM(out_ch)
        self.block = ResidualConvBlock(out_ch * 2, out_ch, norm=norm, dropout=dropout)

    def forward(self, x, skip):
        x = self.up(x)
        if self.use_attention_gate:
            skip = self.attn(x, skip)
        skip = self.cbam(skip)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)

class BoundaryRefinementModule(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.edge_conv = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        with torch.no_grad():
            sobel = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=torch.float32)
            sobel = sobel.abs() / sobel.abs().sum()
            for i in range(channels):
                self.edge_conv.weight[i, 0] = sobel
        self.attn = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1, bias=False),
            nn.InstanceNorm2d(channels, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.InstanceNorm2d(channels, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv2d(channels, channels, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, feat):
        edges = self.edge_conv(feat)
        attn = self.attn(torch.cat([feat, edges], dim=1))
        return feat * (1 + attn)

class MultiScaleFusion(nn.Module):
    def __init__(self, channels_list, out_channels):
        super().__init__()
        self.proj = nn.ModuleList([nn.Conv2d(ch, out_channels, 1, bias=False) for ch in channels_list])
        self.norm = nn.InstanceNorm2d(out_channels, affine=True)
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, features: List[torch.Tensor]):
        target_size = features[0].shape[2:]
        ups = []
        for conv, feat in zip(self.proj, features):
            proj = conv(feat)
            if proj.shape[2:] != target_size:
                proj = F.interpolate(proj, size=target_size, mode='bilinear', align_corners=False)
            ups.append(proj)
        fused = self.act(self.norm(sum(ups)))
        return fused

class SegUNetV2(nn.Module):
    def __init__(self, in_ch=4, base=48, dim=384, patch=8, depth=4, n_heads=8,
                 num_classes=3, dropout=0.15, norm='instance', deep_supervision=True,
                 multi_scale_fusion=True, boundary_refinement=False,
                 use_multiscale_transformer=False, use_attention_gates=False):
        super().__init__()
        self.deep_supervision = deep_supervision
        self.multi_scale_fusion = multi_scale_fusion
        self.boundary_refinement = boundary_refinement
        self.use_multiscale_transformer = use_multiscale_transformer
        self.e1 = EncoderBlock(in_ch, base, norm=norm)
        self.e2 = EncoderBlock(base, base * 2, norm=norm)
        self.e3 = EncoderBlock(base * 2, base * 4, norm=norm, dropout=dropout)
        self.e4 = EncoderBlock(base * 4, base * 8, norm=norm, dropout=dropout)
        self.bottleneck_conv = conv_norm_act(base * 8, dim, k=1, s=1, p=0, norm=norm)
        if use_multiscale_transformer:
            self.bottleneck = MultiScaleTransformerBottleneck(dim, dim)
            self.post_tr = nn.Conv2d(dim, base * 8, 1, bias=False)
        else:
            self.amt = AdaptiveMaskedTransformer(dim, dim, patch_size=patch, depth=depth, n_heads=n_heads)
            self.tr_upsample = nn.ConvTranspose2d(dim, base * 8, patch, stride=patch, bias=False)
        self.d4 = DecoderBlock(base * 8, base * 8, norm=norm, dropout=dropout, use_attention_gate=use_attention_gates)
        self.d3 = DecoderBlock(base * 8, base * 4, norm=norm, dropout=dropout, use_attention_gate=use_attention_gates)
        self.d2 = DecoderBlock(base * 4, base * 2, norm=norm, dropout=dropout / 2, use_attention_gate=use_attention_gates)
        self.d1 = DecoderBlock(base * 2, base, norm=norm, dropout=0.0, use_attention_gate=use_attention_gates)
        if multi_scale_fusion:
            self.ms_fusion = MultiScaleFusion([base, base * 2, base * 4, base * 8], base)
            self.fusion_conv = ResidualConvBlock(base * 2, base, norm=norm)
        if boundary_refinement:
            self.boundary = BoundaryRefinementModule(base)
        self.head = nn.Conv2d(base, num_classes, 1)
        if deep_supervision:
            self.aux_head3 = nn.Conv2d(base * 4, num_classes, 1)
            self.aux_head2 = nn.Conv2d(base * 2, num_classes, 1)
            self.aux_head1 = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        s1, x1 = self.e1(x)
        s2, x2 = self.e2(x1)
        s3, x3 = self.e3(x2)
        s4, x4 = self.e4(x3)
        b = self.bottleneck_conv(x4)
        if self.use_multiscale_transformer:
            b = self.post_tr(self.bottleneck(b))
        else:
            b = self.tr_upsample(self.amt(b))
        d4 = self.d4(b, s4)
        d3 = self.d3(d4, s3)
        aux3 = self.aux_head3(d3) if self.deep_supervision else None
        d2 = self.d2(d3, s2)
        aux2 = self.aux_head2(d2) if self.deep_supervision else None
        d1 = self.d1(d2, s1)
        aux1 = self.aux_head1(d1) if self.deep_supervision else None
        if self.multi_scale_fusion:
            fused = self.ms_fusion([d1, d2, d3, d4])
            feat = self.fusion_conv(torch.cat([d1, fused], dim=1))
        else:
            feat = d1
        if self.boundary_refinement:
            feat = self.boundary(feat)
        seg = self.head(feat)
        if self.deep_supervision:
            return seg, [aux3, aux2, aux1]
        return seg

class TInceptionBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        c = out_ch // 4
        self.b1 = nn.Sequential(nn.Conv2d(in_ch, c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.b2 = nn.Sequential(nn.Conv2d(in_ch, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.b3 = nn.Sequential(nn.Conv2d(in_ch, c, (1, 3), padding=(0, 1), bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.b4 = nn.Sequential(nn.Conv2d(in_ch, c, (3, 1), padding=(1, 0), bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(nn.Conv2d(c * 4, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        x = torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)
        return self.fuse(x)

class TInceptionNet(nn.Module):
    def __init__(self, in_ch=1, num_classes=2):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.b1 = TInceptionBlock(64, 128)
        self.b2 = TInceptionBlock(128, 256)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.b1(x)
        x = self.b2(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)

class BrainTumNetV2(nn.Module):
    def __init__(self, cfg: Dict[str, Any]):
        super().__init__()
        mcfg = cfg['model']
        self.num_classes_seg = mcfg['num_classes_seg']
        self.deep_supervision = mcfg.get('deep_supervision', True)
        self.seg = SegUNetV2(
            in_ch=mcfg['in_channels'],
            base=mcfg['base'],
            dim=mcfg['dim'],
            patch=mcfg['patch_size'],
            depth=mcfg['depth'],
            n_heads=mcfg['n_heads'],
            num_classes=mcfg['num_classes_seg'],
            dropout=mcfg.get('dropout', 0.15),
            deep_supervision=self.deep_supervision,
            multi_scale_fusion=mcfg.get('multi_scale_fusion', True),
            boundary_refinement=mcfg.get('boundary_refinement', False),
            use_multiscale_transformer=mcfg.get('use_multiscale_transformer', False),
            use_attention_gates=mcfg.get('use_attention_gates', False)
        )
        in_ch = mcfg['in_channels']
        self.reduce = nn.Conv2d(in_ch, 1, 1, bias=False) if in_ch > 1 else nn.Identity()
        self.cls_backbone = TInceptionNet(in_ch=1, num_classes=mcfg['num_classes_cls'])
        self.roi_stop_grad = mcfg.get('roi_stop_grad', True)

    def forward(self, x):
        seg_out = self.seg(x)
        if self.deep_supervision:
            seg_logits, aux = seg_out
        else:
            seg_logits = seg_out
            aux = None
        if self.num_classes_seg == 1:
            seg_prob = torch.sigmoid(seg_logits)
        else:
            seg_prob = torch.softmax(seg_logits, dim=1)[:, 1:, :, :].sum(dim=1, keepdim=True)
        roi_input = self.reduce(x)
        roi = roi_input * (seg_prob.detach() if self.roi_stop_grad else seg_prob)
        cls_logits = self.cls_backbone(roi)
        if self.deep_supervision:
            return seg_logits, cls_logits, aux
        return seg_logits, cls_logits

## 6. Losses và metrics

In [ ]:
def dice_loss_with_logits(logits, target, eps=1e-6):
    logits = logits.float()
    target = target.float()
    pred = torch.sigmoid(logits)
    num = 2 * (pred * target).sum(dim=(2, 3))
    den = pred.pow(2).sum(dim=(2, 3)) + target.pow(2).sum(dim=(2, 3)) + eps
    dice = 1 - (num + eps) / den
    return dice.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        if isinstance(alpha, (list, tuple)):
            self.alpha_pos = float(alpha[1] if len(alpha) > 1 else alpha[0])
            self.alpha_neg = float(alpha[0])
        else:
            self.alpha_pos = float(alpha)
            self.alpha_neg = 1.0 - float(alpha)
        self.gamma = gamma

    def forward(self, logits, target):
        logits = logits.float()
        probs = torch.sigmoid(logits).clamp_min(1e-6).clamp_max(1-1e-6)
        pt = torch.where(target == 1, probs, 1 - probs)
        alpha = torch.where(target == 1, logits.new_tensor(self.alpha_pos), logits.new_tensor(self.alpha_neg))
        focal = alpha * (1 - pt).pow(self.gamma) * (-torch.log(pt))
        return focal.mean()

class DiceCELoss(nn.Module):
    def __init__(self, pos_weight=None):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight])) if pos_weight else nn.BCEWithLogitsLoss()

    def forward(self, seg_logits, seg_mask):
        return dice_loss_with_logits(seg_logits, seg_mask) + self.bce(seg_logits, seg_mask)

class DiceFocalLoss(nn.Module):
    def __init__(self, focal_alpha=0.25, focal_gamma=2.0):
        super().__init__()
        self.focal = FocalLoss(focal_alpha, focal_gamma)

    def forward(self, seg_logits, seg_mask):
        return dice_loss_with_logits(seg_logits, seg_mask) + self.focal(seg_logits, seg_mask)

class BoundaryLoss(nn.Module):
    def compute_distance_map(self, mask):
        mask_np = mask.detach().cpu().numpy().astype(bool)
        dist_maps = []
        for b in mask_np:
            if b.any():
                pos = distance_transform_edt(b)
                neg = distance_transform_edt(~b)
                dist_maps.append((neg - pos).astype(np.float32))
            else:
                dist_maps.append(np.zeros_like(b, dtype=np.float32))
        return torch.from_numpy(np.stack(dist_maps)).unsqueeze(1).to(mask.device)

    def forward(self, logits, target):
        prob = torch.sigmoid(logits)
        dist = self.compute_distance_map(target)
        return ((prob - target) * dist).abs().mean()

def multiclass_dice_loss(logits, targets, num_classes, ignore_background=True, smooth=1.0):
    probs = torch.softmax(logits, dim=1)
    targets = targets.squeeze(1).long()
    one_hot = F.one_hot(targets, num_classes=num_classes).permute(0, 3, 1, 2).float()
    start = 1 if ignore_background else 0
    dice_scores = []
    for c in range(start, num_classes):
        pred_c = probs[:, c]
        target_c = one_hot[:, c]
        intersection = (pred_c * target_c).sum(dim=(1, 2))
        union = pred_c.sum(dim=(1, 2)) + target_c.sum(dim=(1, 2))
        dice_scores.append((2.0 * intersection + smooth) / (union + smooth))
    return 1.0 - torch.stack(dice_scores).mean()

class MulticlassDiceCELoss(nn.Module):
    def __init__(self, num_classes=3, ignore_background=True, class_weights=None):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_background = ignore_background
        if class_weights is not None:
            self.ce = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32))
        else:
            self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        dice = multiclass_dice_loss(logits, targets, self.num_classes, self.ignore_background)
        ce = self.ce(logits, targets.squeeze(1).long())
        return dice + ce

class MulticlassFocalLoss(nn.Module):
    def __init__(self, num_classes=3, alpha=None, gamma=2.0, ignore_background=True):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma
        if alpha is None:
            alpha = [1.0] * num_classes
        if ignore_background and num_classes > 0:
            alpha[0] = 0.0
        self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        targets = targets.squeeze(1).long()
        probs_flat = probs.permute(0, 2, 3, 1).reshape(-1, self.num_classes)
        targets_flat = targets.reshape(-1)
        idx = torch.arange(targets_flat.numel(), device=targets_flat.device)
        pt = probs_flat[idx, targets_flat].clamp_min(1e-7)
        alpha = self.alpha.to(logits.device)[targets_flat]
        loss = -alpha * (1 - pt).pow(self.gamma) * torch.log(pt)
        return loss.mean()

class MulticlassDiceFocalLoss(nn.Module):
    def __init__(self, num_classes=3, ignore_background=True, focal_alpha=None, focal_gamma=2.0):
        super().__init__()
        self.dice = MulticlassDiceCELoss(num_classes, ignore_background)
        self.focal = MulticlassFocalLoss(num_classes, focal_alpha, focal_gamma, ignore_background)

    def forward(self, logits, targets):
        return self.dice(logits, targets) + self.focal(logits, targets)

class MultiTaskLoss(nn.Module):
    def __init__(self, cfg: Dict[str, Any]):
        super().__init__()
        tcfg = cfg['train']
        mcfg = cfg['model']
        self.seg_w = tcfg['seg_loss_weight']
        self.cls_w = tcfg['cls_loss_weight']
        self.boundary_w = tcfg.get('boundary_weight', 0.0)
        self.loss_type = tcfg['loss_type']
        num_classes = mcfg['num_classes_seg']
        ignore_bg = tcfg.get('ignore_background', True)
        if num_classes > 1:
            if self.loss_type == 'dice_focal':
                self.seg_loss = MulticlassDiceFocalLoss(num_classes, ignore_bg, tcfg.get('focal_alpha'))
            else:
                self.seg_loss = MulticlassDiceCELoss(num_classes, ignore_bg, tcfg.get('class_weights'))
        else:
            if self.loss_type == 'dice_focal':
                self.seg_loss = DiceFocalLoss(tcfg.get('focal_alpha', 0.25), tcfg.get('focal_gamma', 2.0))
            elif self.loss_type == 'dice_ce_weighted':
                self.seg_loss = DiceCELoss(tcfg.get('pos_weight'))
            else:
                self.seg_loss = DiceCELoss()
        self.cls_loss = nn.CrossEntropyLoss()
        self.boundary = BoundaryLoss() if self.boundary_w > 0 else None

    def forward(self, seg_logits, seg_mask, cls_logits, cls_label, aux_outputs=None):
        seg_loss = self.seg_loss(seg_logits, seg_mask)
        if self.boundary is not None:
            seg_loss = seg_loss + self.boundary_w * self.boundary(seg_logits, seg_mask)
        cls_loss = torch.tensor(0.0, device=seg_logits.device)
        if cls_logits is not None and self.cls_w > 0:
            cls_loss = self.cls_loss(cls_logits, cls_label)
        total = self.seg_w * seg_loss + self.cls_w * cls_loss
        if aux_outputs is not None:
            weights = [0.5, 0.25, 0.125]
            for w, aux in zip(weights, aux_outputs):
                total = total + w * self.seg_loss(aux, seg_mask)
        return total, seg_loss.detach(), cls_loss.detach()

In [ ]:
def compute_hausdorff_distance_95(pred: np.ndarray, target: np.ndarray) -> float:
    pred_pts = np.argwhere(pred > 0)
    target_pts = np.argwhere(target > 0)
    if len(pred_pts) == 0 or len(target_pts) == 0:
        return float('inf')
    from scipy.spatial.distance import cdist
    dists = cdist(pred_pts, target_pts)
    forward = np.percentile(dists.min(axis=1), 95)
    backward = np.percentile(dists.min(axis=0), 95)
    return float(max(forward, backward))

class MulticlassMetricsAccumulator:
    def __init__(self, num_classes: int = 3, compute_hd95: bool = True):
        self.num_classes = num_classes
        self.compute_hd95 = compute_hd95
        self.reset()

    def reset(self):
        self.inter = {k: 0.0 for k in ['WT', 'TC', 'ED']}
        self.union = {k: 0.0 for k in ['WT', 'TC', 'ED']}
        self.hd95_sum = {k: 0.0 for k in ['WT', 'TC', 'ED']}
        self.hd95_count = {k: 0 for k in ['WT', 'TC', 'ED']}

    def update(self, logits: torch.Tensor, target: torch.Tensor):
        pred_classes = torch.argmax(logits, dim=1)
        tgt = target.squeeze(1)
        regions = {
            'TC': (pred_classes == 1, tgt == 1),
            'ED': (pred_classes == 2, tgt == 2),
        }
        regions['WT'] = ((pred_classes >= 1), (tgt >= 1))
        for name, (pred_mask, tgt_mask) in regions.items():
            inter = (pred_mask & tgt_mask).sum().item()
            union = pred_mask.sum().item() + tgt_mask.sum().item()
            self.inter[name] += inter
            self.union[name] += union
            if self.compute_hd95:
                pred_np = pred_mask.cpu().numpy()
                tgt_np = tgt_mask.cpu().numpy()
                for i in range(pred_np.shape[0]):
                    if pred_np[i].any() and tgt_np[i].any():
                        hd95 = compute_hausdorff_distance_95(pred_np[i], tgt_np[i])
                        if np.isfinite(hd95):
                            self.hd95_sum[name] += hd95
                            self.hd95_count[name] += 1

    def get_metrics(self) -> Dict[str, float]:
        eps = 1e-6
        metrics = {}
        for name in ['WT', 'TC', 'ED']:
            dice = (2 * self.inter[name] + eps) / (self.union[name] + eps)
            iou = (self.inter[name] + eps) / (self.union[name] - self.inter[name] + eps)
            hd = self.hd95_sum[name] / self.hd95_count[name] if self.hd95_count[name] > 0 else -1.0
            metrics[f"{name}_dice"] = dice
            metrics[f"{name}_iou"] = iou
            metrics[f"{name}_hd95"] = hd
        metrics['mean_dice'] = (metrics['WT_dice'] + metrics['TC_dice'] + metrics['ED_dice']) / 3.0
        metrics['mean_iou'] = (metrics['WT_iou'] + metrics['TC_iou'] + metrics['ED_iou']) / 3.0
        vals = [metrics[f"{name}_hd95"] for name in ['WT', 'TC', 'ED'] if metrics[f"{name}_hd95"] >= 0]
        metrics['mean_hd95'] = float(np.mean(vals)) if vals else -1.0
        return metrics

def get_multiclass_predictions(logits: torch.Tensor) -> torch.Tensor:
    return torch.argmax(logits, dim=1, keepdim=True)

def visualize_multiclass_prediction(labels: torch.Tensor) -> torch.Tensor:
    if labels.ndim == 4:
        labels = labels.squeeze(1)
    B, H, W = labels.shape
    rgb = torch.zeros(B, 3, H, W, device=labels.device, dtype=torch.float32)
    rgb[:, 0][labels == 1] = 1.0
    rgb[:, 1][labels == 2] = 1.0
    return rgb

## 7. Utilities: logger, checkpoint

In [ ]:
def ensure_dir(path: Path):
    Path(path).mkdir(parents=True, exist_ok=True)

def save_ckpt(model: nn.Module, path: Path):
    ensure_dir(path.parent)
    torch.save(model.state_dict(), path)

def save_training_state(path: Path, epoch: int, model, optimizer, scheduler, scaler,
                       best_iou: float, best_epoch: int, cfg: Dict[str, Any], fold: int):
    ensure_dir(path.parent)
    state = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'best_iou': best_iou,
        'best_iou_epoch': best_epoch,
        'config': cfg,
        'fold': fold,
    }
    if scheduler is not None:
        state['scheduler'] = scheduler.state_dict()
    if scaler is not None:
        state['scaler'] = scaler.state_dict()
    torch.save(state, path)

def load_training_state(path: Path, model, optimizer, scheduler=None, scaler=None, map_location='cpu'):
    state = torch.load(path, map_location=map_location)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    if scheduler is not None and 'scheduler' in state:
        scheduler.load_state_dict(state['scheduler'])
    if scaler is not None and 'scaler' in state:
        scaler.load_state_dict(state['scaler'])
    return state

class TrainingLogger:
    def __init__(self, log_dir: Path, exp_name: str, fold: int):
        ensure_dir(log_dir)
        self.path = log_dir / f"{exp_name}_fold{fold}.log"
        with self.path.open('w') as f:
            f.write(f"BrainTumNet log
Experiment: {exp_name} Fold: {fold}
")

    def info(self, msg: str):
        print(msg)
        with self.path.open('a') as f:
            f.write(msg + "
")

    def epoch_start(self, epoch: int, total: int, phase: str):
        self.info(f"--- Epoch {epoch+1}/{total} [{phase}] ---")

    def epoch_end(self, epoch: int, total: int, metrics: Dict[str, float], phase: str):
        metrics_str = ', '.join(f"{k}: {v:.4f}" for k, v in metrics.items())
        self.info(f"Epoch {epoch+1}/{total} [{phase}] {metrics_str}")

    def best_checkpoint(self, metric_name: str, metric_value: float, epoch: int):
        self.info(f"NEW BEST {metric_name.upper()} = {metric_value:.4f} at epoch {epoch+1}")

class MetricsLogger:
    def __init__(self, log_dir: Path, exp_name: str, fold: int):
        ensure_dir(log_dir)
        self.rows = []
        self.csv_path = log_dir / f"metrics_{exp_name}_fold{fold}.csv"

    def log_epoch(self, epoch: int, metrics: Dict[str, float]):
        row = {'epoch': epoch}
        row.update(metrics)
        self.rows.append(row)
        df = pd.DataFrame(self.rows)
        df.to_csv(self.csv_path, index=False)

    def get_best_metrics(self):
        best = {}
        for key in self.rows[0].keys():
            if key == 'epoch':
                continue
            values = [(row[key], row['epoch']) for row in self.rows if key in row]
            if 'loss' in key:
                best[key] = min(values)[0]
            else:
                best[key] = max(values)[0]
        return best

## 8. Training engine

In [ ]:
try:
    import torchvision
except ImportError:
    torchvision = None


def _cosine_lr_with_warmup(optimizer, base_lr, step, total_steps, warmup_steps=500, min_lr=1e-6):
    if step < warmup_steps:
        lr = base_lr * step / max(1, warmup_steps)
    else:
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr


def centralized_gradient(optimizer):
    for group in optimizer.param_groups:
        for p in group['params']:
            if p.grad is None or p.grad.ndim <= 1:
                continue
            dims = tuple(range(1, p.grad.ndim))
            p.grad.sub_(p.grad.mean(dim=dims, keepdim=True))


class DeepSupervisionScheduler:
    def __init__(self, initial_weight=0.5, final_weight=0.1, total_epochs=400):
        self.initial = initial_weight
        self.final = final_weight
        self.total = total_epochs

    def get_weight(self, epoch):
        progress = min(epoch / self.total, 1.0)
        return self.initial + (self.final - self.initial) * progress


def prepare_artifact_dirs(cfg: Dict[str, Any]):
    logging_cfg = cfg['logging']
    exp_name = logging_cfg.get('exp_name', 'braintumnet')
    log_dir = Path(logging_cfg['log_dir']) / cfg['model']['model_type'] / exp_name
    out_dir = Path(logging_cfg['out_dir']) / cfg['model']['model_type'] / exp_name
    save_dir = Path(logging_cfg['save_dir']) / cfg['model']['model_type'] / exp_name
    for path in [log_dir, out_dir, save_dir]:
        ensure_dir(path)
    logging_cfg.update({'log_dir': str(log_dir), 'out_dir': str(out_dir), 'save_dir': str(save_dir)})
    return log_dir, out_dir, save_dir


def build_model(cfg: Dict[str, Any]):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = BrainTumNetV2(cfg).to(device)
    if cfg['train'].get('channels_last'):
        model = model.to(memory_format=torch.channels_last)
    return model, device


def train_one_fold(cfg: Dict[str, Any], fold: int = 0, resume_from: Optional[Path] = None):
    cfg = json.loads(json.dumps(cfg))  # deep copy to avoid side-effects
    cfg['data']['fold'] = fold
    log_dir, out_dir, save_dir = prepare_artifact_dirs(cfg)
    logger = TrainingLogger(log_dir, cfg['logging']['exp_name'], fold)
    metrics_logger = MetricsLogger(log_dir, cfg['logging']['exp_name'], fold)
    train_loader, val_loader = build_dataloaders(cfg, fold)
    model, device = build_model(cfg)
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    logger.info(f"Model params: {total_params:.2f}M")
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['train']['lr'], weight_decay=cfg['train']['weight_decay'], fused=cfg['train'].get('optimizer_fused', False))
    scheduler = None
    if cfg['train']['scheduler'] == 'plateau':
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-7)
    elif cfg['train']['scheduler'] == 'onecycle':
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=cfg['train']['lr'], epochs=cfg['train']['epochs'], steps_per_epoch=len(train_loader))
    elif cfg['train']['scheduler'] == 'cosine_restarts':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=cfg['train'].get('T_0', 50), T_mult=cfg['train'].get('T_mult', 2), eta_min=cfg['train'].get('min_lr', 1e-5))
    criterion = MultiTaskLoss(cfg)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg['train'].get('amp', False) and cfg['train'].get('amp_dtype', 'float16') == 'float16')
    writer = SummaryWriter(out_dir) if HAS_TENSORBOARD and cfg['logging'].get('use_tensorboard', True) else None
    start_epoch = 0
    best_iou = -1.0
    best_epoch = 0
    total_steps = cfg['train']['epochs'] * max(1, len(train_loader))
    step = 0
    if resume_from and Path(resume_from).exists():
        state = load_training_state(Path(resume_from), model, optimizer, scheduler, scaler, map_location=device)
        start_epoch = state['epoch'] + 1
        best_iou = state.get('best_iou', best_iou)
        best_epoch = state.get('best_iou_epoch', best_epoch)
        logger.info(f"Resumed from {resume_from} at epoch {start_epoch}")
    ds_sched = DeepSupervisionScheduler(cfg['train'].get('aux_weight_initial', 0.5), cfg['train'].get('aux_weight_final', 0.1), cfg['train']['epochs']) if cfg['train'].get('aux_weight_initial') else None
    for epoch in range(start_epoch, cfg['train']['epochs']):
        model.train()
        logger.epoch_start(epoch, cfg['train']['epochs'], 'TRAIN')
        running_loss = 0.0
        accum_steps = cfg['train'].get('grad_accum_steps', 1)
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", ncols=120)
        for batch_idx, batch in enumerate(pbar):
            img = batch['image'].to(device)
            mask = batch['mask'].to(device)
            label = batch['label'].to(device)
            if cfg['train'].get('channels_last'):
                img = img.to(memory_format=torch.channels_last)
            with torch.cuda.amp.autocast(enabled=cfg['train'].get('amp', False), dtype=torch.float16 if cfg['train'].get('amp_dtype') == 'float16' else torch.bfloat16):
                output = model(img)
                if isinstance(output, tuple):
                    if len(output) == 3:
                        seg, cls, aux = output
                    elif len(output) == 2:
                        seg, cls = output
                        aux = None
                    else:
                        seg, cls, aux = output, None, None
                else:
                    seg, cls, aux = output, None, None
                if ds_sched is not None and aux is not None:
                    aux_weights = [ds_sched.get_weight(epoch) * w for w in [1.0, 0.5, 0.25]]
                    aux_scaled = [w * a for w, a in zip(aux_weights, aux)]
                    loss, seg_loss, cls_loss = criterion(seg, mask, cls, label, aux_scaled)
                else:
                    loss, seg_loss, cls_loss = criterion(seg, mask, cls, label, aux)
                loss = loss / accum_steps
            scaler.scale(loss).backward()
            if (batch_idx + 1) % accum_steps == 0:
                if cfg['train'].get('optimizer', 'adamw') == 'adamw':
                    centralized_gradient(optimizer)
                if cfg['train'].get('grad_clip_norm', 0) > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['train']['grad_clip_norm'])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if cfg['train']['scheduler'] == 'onecycle' and scheduler is not None:
                    scheduler.step()
                if cfg['train']['scheduler'] == 'cosine_restarts' and scheduler is not None:
                    scheduler.step()
            if cfg['train']['scheduler'] == 'cosine':
                _cosine_lr_with_warmup(optimizer, cfg['train']['lr'], step, total_steps, cfg['train'].get('warmup_steps', 500), cfg['train'].get('min_lr', 1e-6))
            running_loss += loss.item() * accum_steps
            step += 1
            pbar.set_postfix({'loss': f"{loss.item() * accum_steps:.4f}", 'lr': f"{optimizer.param_groups[0]['lr']:.2e}"})
        avg_train_loss = running_loss / len(train_loader)
        # Validation
        model.eval()
        metrics_acc = MulticlassMetricsAccumulator(cfg['model']['num_classes_seg']) if cfg['model']['num_classes_seg'] > 1 else None
        total_inter = total_union = 0.0
        hd95_sum = hd95_count = 0
        cls_acc = []
        with torch.inference_mode():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", ncols=120):
                img = batch['image'].to(device)
                mask = batch['mask'].to(device)
                label = batch['label'].to(device)
                if cfg['train'].get('channels_last'):
                    img = img.to(memory_format=torch.channels_last)
                output = model(img)
                if isinstance(output, tuple):
                    seg = output[0]
                    cls = output[1] if len(output) > 1 else None
                else:
                    seg, cls = output, None
                if metrics_acc is not None:
                    metrics_acc.update(seg, mask)
                else:
                    preds = (torch.sigmoid(seg) > 0.5).float()
                    total_inter += (preds * mask).sum().item()
                    total_union += preds.sum().item() + mask.sum().item()
                    pred_np = preds.cpu().numpy()
                    mask_np = mask.cpu().numpy()
                    for i in range(pred_np.shape[0]):
                        if pred_np[i].any() and mask_np[i].any():
                            hd = compute_hausdorff_distance_95(pred_np[i, 0], mask_np[i, 0])
                            if np.isfinite(hd):
                                hd95_sum += hd
                                hd95_count += 1
                if cls is not None:
                    cls_acc.append((cls.argmax(1) == label).float().mean().item())
        if metrics_acc is not None:
            metrics = metrics_acc.get_metrics()
            dice_m = metrics['mean_dice']
            iou_m = metrics['mean_iou']
            hd95_m = metrics['mean_hd95']
        else:
            eps = 1e-6
            dice_m = (2 * total_inter) / (total_union + eps)
            iou_m = total_inter / (total_union - total_inter + eps)
            hd95_m = hd95_sum / hd95_count if hd95_count > 0 else -1.0
            metrics = {'WT_dice': dice_m, 'WT_iou': iou_m, 'WT_hd95': hd95_m, 'TC_dice': dice_m, 'TC_iou': iou_m, 'TC_hd95': hd95_m, 'ED_dice': dice_m, 'ED_iou': iou_m, 'ED_hd95': hd95_m}
        acc_m = float(np.mean(cls_acc)) if cls_acc else 0.0
        logger.epoch_end(epoch, cfg['train']['epochs'], {
            'train_loss': avg_train_loss,
            'val_dice': dice_m,
            'val_iou': iou_m,
            'val_hd95': hd95_m,
            'val_acc': acc_m,
            'lr': optimizer.param_groups[0]['lr']
        }, 'SUMMARY')
        metrics_dict = {'train_loss': avg_train_loss, 'val_dice': dice_m, 'val_iou': iou_m, 'val_hd95': hd95_m, 'val_acc': acc_m, 'learning_rate': optimizer.param_groups[0]['lr']}
        metrics_logger.log_epoch(epoch, metrics_dict)
        if writer:
            writer.add_scalar('train/loss', avg_train_loss, epoch)
            writer.add_scalar('val/dice', dice_m, epoch)
            writer.add_scalar('val/iou', iou_m, epoch)
            if hd95_m >= 0:
                writer.add_scalar('val/hd95', hd95_m, epoch)
            writer.add_scalar('val/cls_acc', acc_m, epoch)
        if cfg['train']['scheduler'] == 'plateau' and scheduler is not None:
            scheduler.step(iou_m)
        if iou_m > best_iou:
            best_iou = iou_m
            best_epoch = epoch
            best_path = Path(cfg['logging']['save_dir']) / f"braintumnet_best_fold{fold}.pth"
            save_ckpt(model, best_path)
            logger.best_checkpoint('IoU', best_iou, epoch)
        last_ckpt = Path(cfg['logging']['save_dir']) / f"last_fold{fold}.pth"
        active_scheduler = scheduler if cfg['train']['scheduler'] in ['plateau', 'onecycle', 'cosine_restarts'] else None
        save_training_state(last_ckpt, epoch, model, optimizer, active_scheduler, scaler, best_iou, best_epoch, cfg, fold)
        if epoch - best_epoch >= cfg['train'].get('early_stop_patience', 30):
            logger.info("Early stopping triggered")
            break
    if writer:
        writer.close()
    logger.info(f"Training done. Best IoU {best_iou:.4f} at epoch {best_epoch+1}")
    return best_iou

## 9. Ví dụ sử dụng

In [ ]:
# Ví dụ: chạy preprocessing toàn bộ BraTS (mặc định giữ nguyên tất cả lát cắt)
# uncomment để chạy thật
# if False:
#     pre_cfg = PreprocessConfig(
#         nifti_dir=Path('braintumnet/data/raw/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'),
#         out_dir=PROCESSED_DIR,
#         img_size=256,
#         slices_per_case=None,
#         tumor_ratio=0.7,
#         num_folds=5
#     )
#     preprocess_brats_to_png(pre_cfg)

# Ví dụ: convert PNG → LMDB
# if False:
#     lmdb_cfg = LmdbConversionConfig(
#         input_dir=PROCESSED_DIR,
#         output_dir=LMDB_DIR,
#         map_size_gb=60,
#         verify=True
#     )
#     convert_png_to_lmdb(lmdb_cfg)

# Ví dụ: huấn luyện fold 0
# if False:
#     cfg = build_phase2_config(backend='lmdb')
#     best_iou = train_one_fold(cfg, fold=0)
#     print('Best IoU:', best_iou)